# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Maryam/Aleeza-ML-Internship-WEEK1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Research Question

Can observable search and content signals be used to rank pages that are potential content-refresh opportunities better than a simple hand-written rule?

## Lane

Refresh / Content Opportunity Scoring

## Decision Supported

The goal is to help content teams prioritize which pages should be reviewed first for a possible content refresh.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

This project uses the FlyRank Internship Warehouse dataset hosted on Hugging Face.

### Tables

The main table is `fact_content_daily_performance`, which contains daily search and analytics performance for content items.

The `dim_content` table is used for content-level metadata such as content age, content type, search volume, and word count.

### Development Window

The main development data uses a mid-panel month such as March 2026.

The final month is treated as a sealed test period rather than being used to develop the label or rule.

### Excluded Data

Client names, domains, URLs, private queries, credentials, and other identifying information are excluded.

Future-window information is also excluded from the features because it would cause data leakage.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

The analysis will use observable signals that are available at the decision moment.

Candidate features include search impressions, clicks, average search position, analytics activity, content age, days since last update, search volume, and word count.

A future-window outcome will be used as the prediction target so that the model learns from information available before the outcome occurs.

The existing hand-written refresh rule from Week 4 will be used as the baseline.

The machine-learning model will be evaluated against this baseline using the same validation split.

Leakage checks will ensure that future performance information and label-derived variables are not used as model features.

In [2]:
import os
import duckdb
from google.colab import userdata

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

# Hugging Face dataset
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

# Create DuckDB connection
con = duckdb.connect()

# Configure Hugging Face authentication
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB connection ready.")
print("HF dataset:", HF_DATASET)

Token loaded: True
DuckDB connection ready.
HF dataset: hf://datasets/FlyRank/internship-warehouse


In [3]:
q = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ first_date │ last_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



In [6]:
features_df = march_features.df()

print("Rows:", len(features_df))
print("Columns:", features_df.columns.tolist())

features_df.head(10)

Rows: 10000
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'pageviews', 'engaged_sessions']


,client_hash_id,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,36.774194,0.064516,4.394234,0.000000,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1.838710,0.000000,2.714744,0.000000,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,4.806452,0.000000,6.481453,0.363636,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,45.838710,0.193548,6.320337,1.090909,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,89.354839,0.516129,4.459107,0.272727,0.0
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,1.548387,0.000000,14.753175,0.000000,0.0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,4.838710,0.032258,6.341880,0.090909,0.0
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,2.161290,0.000000,12.791667,0.000000,0.0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,195.096774,0.741935,4.950311,0.727273,0.0
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,7.193548,0.000000,50.390299,0.090909,0.0


In [7]:
features_df = march_features.df()

print("Rows:", len(features_df))
print("Columns:", features_df.columns.tolist())

features_df.head(10)

Rows: 10000
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'pageviews', 'engaged_sessions']


,client_hash_id,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,5.838710,0.000000,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,1.483871,0.032258,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,29.000000,0.032258,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,1.096774,0.000000,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,100.258065,0.000000,6.969536,NaN,NaN
5,client_62f4a7e64f5e0096,content_d49a012dcb924e31,10.612903,0.000000,5.177774,NaN,NaN
6,client_62f4a7e64f5e0096,content_614baf2af4330bd7,24.903226,0.032258,4.685335,NaN,NaN
7,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,4.322581,0.000000,4.627228,NaN,NaN
8,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,0.451613,0.000000,4.266667,NaN,NaN
9,client_62f4a7e64f5e0096,content_225dc9235023be5f,15.741935,0.032258,17.148172,NaN,NaN


In [8]:
features_df.head(10)

,client_hash_id,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,5.838710,0.000000,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,1.483871,0.032258,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,29.000000,0.032258,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,1.096774,0.000000,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,100.258065,0.000000,6.969536,NaN,NaN
5,client_62f4a7e64f5e0096,content_d49a012dcb924e31,10.612903,0.000000,5.177774,NaN,NaN
6,client_62f4a7e64f5e0096,content_614baf2af4330bd7,24.903226,0.032258,4.685335,NaN,NaN
7,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,4.322581,0.000000,4.627228,NaN,NaN
8,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,0.451613,0.000000,4.266667,NaN,NaN
9,client_62f4a7e64f5e0096,content_225dc9235023be5f,15.741935,0.032258,17.148172,NaN,NaN


In [9]:
content_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS impressions,
    AVG(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS pageviews,
    AVG(ga4_engaged_sessions) AS engaged_sessions
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
LIMIT 10000
""")

features_df = content_features.df()
print("Performance features:", features_df.shape)

Performance features: (10000, 7)


In [10]:
content_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_DATASET}/dim_content.parquet'
)
""")

content_schema.show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Results

This section will compare the machine-learning model with the hand-written baseline on the same validation split.

The final comparison will report the selected ranking metric and include supporting charts.

## 5. Limitations

*What this work cannot claim.*

## Limitations

This analysis identifies patterns and ranking opportunities rather than causal effects.

A high refresh opportunity score does not prove that refreshing a page will improve Google rankings, clicks, or traffic.

The model is intended as decision support for prioritizing human review, not as an automatic publishing or content-removal system.

The analysis is also limited by the available warehouse fields, data coverage, and the chosen time windows.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked Recommendations

The final model will produce a ranked list of content items with an opportunity score, reason code, and recommended action.

The recommendations will be used to prioritize human review for possible content refreshes.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Artifacts

The final paper will include:

- Model versus baseline performance
- Model evaluation metrics
- Feature importance or model interpretation
- Ranked recommendation examples
- Supporting charts

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
